# Phase 3 Overlay Application Slice

This notebook proves explicit overlay application, overlay-aware validation, and report output that distinguishes overlay-sourced additions.

In [1]:
from pathlib import Path
import json
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from onto_canon6.pipeline import OverlayApplicationService, ReviewService  # noqa: E402
from onto_canon6.surfaces import ReviewReportService  # noqa: E402

In [2]:
tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "review.sqlite3"
overlay_root = Path(tmp_dir.name) / "ontology_overlays"
review_service = ReviewService(
    db_path=db_path,
    overlay_root=overlay_root,
    default_acceptance_policy="record_only",
)
overlay_service = OverlayApplicationService(db_path=db_path, overlay_root=overlay_root)
report_service = ReviewReportService(review_service=review_service, overlay_service=overlay_service)
{"db_path": str(db_path), "overlay_root": str(overlay_root)}

{'db_path': '/tmp/tmpkrdwj6jq/review.sqlite3',
 'overlay_root': '/tmp/tmpkrdwj6jq/ontology_overlays'}

In [3]:
initial = review_service.submit_candidate_assertion(
    payload={
        "predicate": "oc:overlay_demo_predicate",
        "roles": {
            "subject": [{"entity_id": "ent:subject:1", "entity_type": "oc:person"}],
        },
    },
    profile_id="psyop_seed",
    profile_version="0.1.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase3/initial-candidate",
    source_label="phase3 initial candidate",
    source_metadata={"slice": "phase3", "case": "initial"},
)
initial.model_dump()

{'candidate': {'candidate_id': 'cand_571f137923304214a24e0173',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'validation_status': 'needs_review',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:e428e19df9f675d85dbb2ed701f3a7d93bea0149ce8c0a259185f12bc28fd3e5',
  'payload': {'predicate': 'oc:overlay_demo_predicate',
   'roles': {'subject': [{'entity_id': 'ent:subject:1',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'normalized_payload': {'predicate': 'oc:overlay_demo_predicate',
   'roles': {'subject': [{'entity_id': 'ent:subject:1',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'validation': {'hard_errors': (),
   'soft_violations': ({'code': 'oc:profile_unknown_predicate',
     'message': "predicate 'oc:overlay_demo_predicate' is not declared in profile psyop_seed; mixed ontology mode routes it as a proposal",
     'path': 'predicate',
     'severity': 'soft'},),
   'type_checks_total': 0,
   'type_checks_

In [4]:
reviewed_proposal = review_service.review_proposal(
    proposal_id=initial.proposals[0].proposal_id,
    decision="accepted",
    actor_id="notebook:reviewer",
    acceptance_policy="apply_to_overlay",
    note_text="Apply this local predicate to the overlay.",
)
reviewed_proposal.model_dump()

{'proposal_id': 'prop_45667b161d156625e4e1dfd1',
 'proposal_key': 'sha256:45667b161d156625e4e1dfd1012eb2ef40ee125897a962b4ae6fdb3dd6f00c3e',
 'proposal_kind': 'predicate',
 'proposed_value': 'oc:overlay_demo_predicate',
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'target_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
  'pack_version': '0.1.0'},
 'reason': "default ontology mode 'mixed' resolved action 'propose' for predicate",
 'status': 'accepted',
 'application_status': 'pending_overlay_apply',
 'details': {'candidate_source_kind': 'notebook',
  'source': 'validate_assertion_payload'},
 'candidate_ids': ('cand_571f137923304214a24e0173',),
 'created_at': '2026-03-17T21:27:56.963934Z',
 'review': {'decision_id': 'dec_ec2020f277dc455d9a276457',
  'proposal_id': 'prop_45667b161d156625e4e1dfd1',
  'decision': 'accepted',
  'actor_id': 'notebook:reviewer',
  'note_text': 'Apply this local predicate to the overlay.',
  'acceptance_policy': 'apply_to_overlay'

In [5]:
application = overlay_service.apply_proposal_to_overlay(
    proposal_id=reviewed_proposal.proposal_id,
    applied_by="notebook:reviewer",
)
application.model_dump()

{'application_id': 'oapp_aab08127168b4501bd048147',
 'proposal_id': 'prop_45667b161d156625e4e1dfd1',
 'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
 'overlay_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
  'pack_version': '0.1.0'},
 'proposal_kind': 'predicate',
 'applied_value': 'oc:overlay_demo_predicate',
 'content_path': '/tmp/tmpkrdwj6jq/ontology_overlays/onto_canon_psyop_seed__overlay/0.1.0/predicate_additions.jsonl',
 'applied_by': 'notebook:reviewer',
 'created_at': '2026-03-17T21:27:56.983987Z'}

In [6]:
overlay_path = Path(application.content_path)
{
    "overlay_exists": overlay_path.exists(),
    "overlay_lines": [json.loads(line) for line in overlay_path.read_text(encoding="utf-8").splitlines() if line.strip()],
}

{'overlay_exists': True,
 'overlay_lines': [{'proposal_id': 'prop_45667b161d156625e4e1dfd1',
   'predicate_id': 'oc:overlay_demo_predicate',
   'base_pack': {'pack_id': 'onto_canon_psyop_seed', 'pack_version': '0.1.0'},
   'overlay_pack': {'pack_id': 'onto_canon_psyop_seed__overlay',
    'pack_version': '0.1.0'},
   'applied_by': 'notebook:reviewer',
   'applied_at': '2026-03-17T21:27:56.983234Z'}]}

In [7]:
revalidated = review_service.submit_candidate_assertion(
    payload={
        "predicate": "oc:overlay_demo_predicate",
        "roles": {
            "subject": [{"entity_id": "ent:subject:2", "entity_type": "oc:person"}],
        },
    },
    profile_id="psyop_seed",
    profile_version="0.1.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase3/revalidated-candidate",
    source_label="phase3 revalidated candidate",
    source_metadata={"slice": "phase3", "case": "revalidated"},
)
revalidated.model_dump()

{'candidate': {'candidate_id': 'cand_549578370c344e50aa0f8ee8',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'validation_status': 'valid',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:d21c645baadb86efd8a2f53769b229ba271fac6ed12067ea4e6ff610e63246f2',
  'payload': {'predicate': 'oc:overlay_demo_predicate',
   'roles': {'subject': [{'entity_id': 'ent:subject:2',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'normalized_payload': {'predicate': 'oc:overlay_demo_predicate',
   'roles': {'subject': [{'entity_id': 'ent:subject:2',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'validation': {'hard_errors': (),
   'soft_violations': (),
   'type_checks_total': 0,
   'type_checks_unknown': 0},
  'proposal_ids': (),
  'provenance': {'submitted_by': 'notebook:analyst',
   'source_artifact': {'source_kind': 'notebook',
    'source_ref': 'notebook://phase3/revalidated-candidate',
    'source_label': 'phase3 revalidated

In [8]:
report = report_service.build_report(profile_id="psyop_seed", profile_version="0.1.0")
report.model_dump()

{'candidates': ({'candidate_id': 'cand_571f137923304214a24e0173',
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'validation_status': 'needs_review',
   'review_status': 'pending_review',
   'payload_hash': 'sha256:e428e19df9f675d85dbb2ed701f3a7d93bea0149ce8c0a259185f12bc28fd3e5',
   'payload': {'predicate': 'oc:overlay_demo_predicate',
    'roles': {'subject': [{'entity_id': 'ent:subject:1',
       'entity_type': 'oc:person',
       'kind': 'entity'}]}},
   'normalized_payload': {'predicate': 'oc:overlay_demo_predicate',
    'roles': {'subject': [{'entity_id': 'ent:subject:1',
       'entity_type': 'oc:person',
       'kind': 'entity'}]}},
   'validation': {'hard_errors': (),
    'soft_violations': ({'code': 'oc:profile_unknown_predicate',
      'message': "predicate 'oc:overlay_demo_predicate' is not declared in profile psyop_seed; mixed ontology mode routes it as a proposal",
      'path': 'predicate',
      'severity': 'soft'},),
    'type_checks_total':

In [9]:
applied_again = overlay_service.apply_proposal_to_overlay(
    proposal_id=reviewed_proposal.proposal_id,
    applied_by="notebook:reviewer",
)
{
    "same_application_id": applied_again.application_id == application.application_id,
    "application_id": applied_again.application_id,
}

{'same_application_id': True,
 'application_id': 'oapp_aab08127168b4501bd048147'}